In [ ]:
!pip install -q transformers datasets trl peft accelerate bitsandbytes

In [ ]:
import json
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from datasets import Dataset
import gc

print(f'GPU: {torch.cuda.is_available()}')
print(f'GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
print(f'GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB' if torch.cuda.is_available() else '')

In [ ]:
training_data = [
    {"instruction": "Привет", "output": "Привет! Чем могу помочь?"},
    {"instruction": "Кто ты?", "output": "Я Dark Chat — AI-ассистент. Могу отвечать на вопросы, генерировать код и помогать с задачами."},
    {"instruction": "Что ты умеешь?", "output": "Я умею: отвечать на вопросы, писать код на разных языках, объяснять сложные вещи простыми словами, помогать с математикой и логикой."},
    {"instruction": "Напиши hello world на Python", "output": "print('Hello, World!')"},
    {"instruction": "Напиши hello world на JavaScript", "output": "console.log('Hello, World!');"},
    {"instruction": "Напиши hello world на Go", "output": "package main\n\nimport \"fmt\"\n\nfunc main() {\n    fmt.Println(\"Hello, World!\")\n}"},
    {"instruction": "Что такое переменная?", "output": "Переменная — это именованная область памяти, в которой хранится значение. Например: x = 5, где x — имя переменной, а 5 — её значение."},
    {"instruction": "Объясни интернет простыми словами", "output": "Интернет — это глобальная сеть компьютеров, соединённых друг с другом. Через интернет можно обмениваться данными."},
    {"instruction": "2+2=", "output": "4"},
    {"instruction": "Что такое AI?", "output": "AI (Artificial Intelligence) — искусственный интеллект. Это компьютерные программы, которые могут учиться, рассуждать и принимать решения, как человек."},
    {"instruction": "Напиши функцию факториала на Python", "output": "def factorial(n):\n    if n <= 1:\n        return 1\n    return n * factorial(n - 1)"},
    {"instruction": "Что такое массив?", "output": "Массив — это структура данных, хранящая список элементов одного типа. Доступ к элементам по индексу: arr[0], arr[1] и т.д."},
    {"instruction": "Расскажи анекдот", "output": "Программист — это тот, кто решает проблему, о которой ты не знал, способом, который ты не понимаешь."},
    {"instruction": "Как создать сайт?", "output": "Для создания сайта нужно: 1) HTML — структура, 2) CSS — стили, 3) JavaScript — логика. Можно использовать фреймворки: React, Vue, Django."},
    {"instruction": "Что такое API?", "output": "API (Application Programming Interface) — интерфейс программного взаимодействия. Это набор правил, по которым программы общаются друг с другом."},
    {"instruction": "Как работает Python?", "output": "Python — интерпретируемый язык. Код читается построчно, преобразуется в байт-код и выполняется виртуальной машиной Python (CPython)."},
    {"instruction": "Что такое база данных?", "output": "База данных — это организованное хранилище информации. Примеры: SQLite, PostgreSQL, MySQL. Данные хранятся в таблицах."},
    {"instruction": "Напиши сортировку пузырьком", "output": "def bubble_sort(arr):\n    n = len(arr)\n    for i in range(n):\n        for j in range(0, n-i-1):\n            if arr[j] > arr[j+1]:\n                arr[j], arr[j+1] = arr[j+1], arr[j]\n    return arr"},
    {"instruction": "Что такое Git?", "output": "Git — система контроля версий. Позволяет отслеживать изменения в коде, работать в команде, откатывать ошибки. GitHub — облачный сервис для Git."},
    {"instruction": "Объясни Docker", "output": "Docker — платформа для контейнеризации. Упакует приложение со всеми зависимостями в контейнер, который работает одинаково на любом сервере."},
    {"instruction": "Что такое REST API?", "output": "REST API — архитектурный стиль для веб-API. Использует HTTP-методы: GET (получить), POST (создать), PUT (обновить), DELETE (удалить)."},
    {"instruction": "Напиши чтение файла на Python", "output": "with open('file.txt', 'r') as f:\n    content = f.read()\nprint(content)"},
    {"instruction": "Что такое рекурсия?", "output": "Рекурсия — функция, которая вызывает сама себя. Пример: factorial(n) = n * factorial(n-1). Должна иметь условие остановки."},
    {"instruction": "Как создать REST API на Python?", "output": "Используй FastAPI:\n\nfrom fastapi import FastAPI\napp = FastAPI()\n\n@app.get('/')\ndef home():\n    return {'message': 'Hello'}"},
    {"instruction": "Что такое SQL?", "output": "SQL — язык запросов для работы с базами данных. Команды: SELECT (выбрать), INSERT (добавить), UPDATE (обновить), DELETE (удалить)."},
    {"instruction": "Напиши чат-бот на Python", "output": "while True:\n    msg = input('Ты: ')\n    if msg == 'привет':\n        print('Бот: Привет!')\n    elif msg == 'пока':\n        print('Бот: Пока!')\n        break\n    else:\n        print('Бот: Не понял')"},
    {"instruction": "Что такое ООП?", "output": "ООП (Объектно-Ориентированное Программирование) — парадигма, где код организован вокруг объектов. Принципы: инкапсуляция, наследование, полиморфизм."},
    {"instruction": "Объясни async/await", "output": "async/await — асинхронное программирование. async def создаёт асинхронную функцию, await ждёт результат. Позволяет выполнять задачи параллельно."},
    {"instruction": "Что такое middleware?", "output": "Middleware — промежуточный слой между запросом и ответом. Обрабатывает данные: аутентификация, логирование, шифрование."},
    {"instruction": "Напиши HTTP-сервер на Python", "output": "from http.server import HTTPServer, SimpleHTTPRequestHandler\n\nserver = HTTPServer(('localhost', 8000), SimpleHTTPRequestHandler)\nprint('Сервер запущен')\nserver.serve_forever()"},
    {"instruction": "Что такое CI/CD?", "output": "CI (Continuous Integration) — автоматическая сборка и тестирование. CD (Continuous Deployment) — автоматический деплой. Инструменты: GitHub Actions, Jenkins."},
]

print(f'Датасет: {len(training_data)} примеров')

In [ ]:
# === МОДЕЛЬ: Mistral 7B с QLoRA (4-bit) ===
# Все техники: QLoRA + LoRA + Flash Attention + Gradient Checkpointing

MODEL_NAME = 'unsloth/mistral-7b-instruct-v0.3-bnb-4bit'

# BitsAndBytes конфигурация (4-bit квантизация)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f'Загружаю {MODEL_NAME} (4-bit QLoRA)...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
)

# Подготовка для QLoRA
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)  # Gradient Checkpointing

print(f'Модель загружена! GPU: {torch.cuda.get_device_name(0)}')
print(f'Память: {torch.cuda.memory_allocated() / 1024**3:.1f} / {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

In [ ]:
# === LoRA конфигурация ===
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                     # Rank (16 — баланс скорость/качество)
    lora_alpha=32,            # Scaling factor
    lora_dropout=0.05,
    bias='none',
    target_modules=[          # Какие слои адаптировать
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
)

# Применяем LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
def format_example(example):
    return {'text': f"### Инструкция:\n{example['instruction']}\n\n### Ответ:\n{example['output']}"}

dataset = Dataset.from_list([format_example(d) for d in training_data])

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=512, padding='max_length')

tokenized = dataset.map(tokenize_function, batched=True, remove_columns=['text'])
split = tokenized.train_test_split(test_size=0.1)

print(f'Train: {len(split["train"])}, Eval: {len(split["test"])}')

In [ ]:
training_args = TrainingArguments(
    output_dir='./darkchat-mistral-qlora',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=50,
    logging_steps=10,
    save_steps=200,
    fp16=True,
    optim='paged_adamw_8bit',
    report_to='none',
    eval_strategy='steps',
    eval_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    gradient_checkpointing=True,  # Gradient Checkpointing
    max_grad_norm=0.3,
    lr_scheduler_type='cosine',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print('=== Все техники активны ===')
print('1. QLoRA (4-bit квантизация)')
print('2. LoRA (r=16, alpha=32)')
print('3. Gradient Checkpointing')
print('4. Paged AdamW 8-bit')
print('5. Mixed Precision (FP16)')
print('\nОбучение...')
trainer.train()
print('Готово!')

In [ ]:
trainer.save_model('./darkchat-mistral-qlora')
tokenizer.save_pretrained('./darkchat-mistral-qlora')
print('Модель сохранена!')

In [ ]:
def generate(prompt):
    inputs = tokenizer(f'### Инструкция:\n{prompt}\n\n### Ответ:\n', return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).split('### Ответ:\n')[-1]

print('=== Тест Mistral 7B + QLoRA ===')
print()
print('Q: Привет!')
print(f'A: {generate("Привет!")}')
print()
print('Q: Напиши hello world на Python')
print(f'A: {generate("Напиши hello world на Python")}')
print()
print('Q: Что такое API?')
print(f'A: {generate("Что такое API?")}')
print()
print('Q: Напиши сортировку пузырьком')
print(f'A: {generate("Напиши сортировку пузырьком")}')

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('darkchat-mistral-qlora', 'zip', './darkchat-mistral-qlora')
files.download('darkchat-mistral-qlora.zip')
print('Скачано!')